# Hugging Face DPO Project: The Gen Academy Brand Voice

**Use case:** fine-tune a model to prefer The Gen Academy's practical, energetic, accessible, builder-focused brand voice.

**What this is:** an LLM DPO project. We start with an instruction-tuned causal language model, give it preference pairs, and train a small LoRA adapter so the LLM assigns higher likelihood to the on-brand answer than the off-brand answer.

**What this is not:** it is not supervised fine-tuning on a single target answer, and it is not training a separate reward model. DPO directly updates the LLM from `prompt`, `chosen`, and `rejected` examples.

**Important dataset note:** the source dataset includes a `preference_reason` field so humans can understand the label. That reason is not passed to DPO training. The trainer only uses the prompt, chosen response, and rejected response.

**Important evaluation note:** preferred answers are intentionally more specific and often longer than rejected hype-heavy answers. Raw summed log-probability penalizes longer completions, so this notebook reports the fairer **length-normalized preference margin** as the main before/after metric.

The comparison uses:

1. Training loss and reward-margin curves.
2. Base vs tuned length-normalized preference-margin charts on held-out examples.
3. A concrete before/after preference example where the tuned adapter moves toward the on-brand answer.


## 1. Setup


In [ ]:
%pip install --quiet pandas datasets transformers trl peft accelerate torch matplotlib huggingface_hub


In [ ]:
import gc
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

SOURCE_DATA_PATH = Path("data/brand_voice_preferences.jsonl")
TRL_DATA_PATH = Path("data/brand_voice_trl_dpo.jsonl")
TRL_TRAIN_PATH = Path("data/brand_voice_trl_dpo_train.jsonl")
TRL_EVAL_PATH = Path("data/brand_voice_trl_dpo_eval.jsonl")


## 2. Inspect the source dataset

The source JSONL is the human-readable version of the demo data. Each row has:

- `prompt`: the task the model should answer.
- `chosen`: the preferred Gen Academy-style response.
- `rejected`: the less preferred response.
- `preference_reason`: a plain-English explanation of why the chosen answer is better.
- `split` and `category`: metadata for evaluation and demo navigation.

The `preference_reason` is useful for explaining the dataset live, but it is intentionally not used by the DPO trainer.


In [ ]:
# This is the readable source dataset. It includes explanation metadata for humans.
source_rows = [
    json.loads(line)
    for line in SOURCE_DATA_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
source_df = pd.DataFrame(source_rows)

print(f"Loaded {len(source_df)} preference pairs")
print(source_df["split"].value_counts().to_string())
source_df.groupby(["split", "category"]).size().unstack(fill_value=0)


In [ ]:
def show_pair(row):
    print(f"{row['id']} | {row['split']} | {row['category']}")
    print("\nTASK")
    print(row["prompt"])
    print("\nCHOSEN")
    print(row["chosen"])
    print("\nREJECTED")
    print(row["rejected"])
    print("\nWHY THIS PAIR IS LABELED THIS WAY")
    print(row["preference_reason"])
    print("\nDPO USES")
    print("prompt + chosen + rejected")
    print("\nDPO DOES NOT USE")
    print("preference_reason, category, or split")


show_pair(source_df.iloc[0])


## 3. Convert to TRL DPO format

TRL's `DPOTrainer` expects each example to contain:

- `prompt`: the conversation context.
- `chosen`: the preferred assistant completion.
- `rejected`: the less preferred assistant completion.

The notebook keeps `id`, `split`, and `category` so we can inspect results later. The `preference_reason` field is dropped before training because DPO should learn from the preference comparison, not from the written explanation.


In [ ]:
trl_rows = [json.loads(line) for line in TRL_DATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
train_rows = [json.loads(line) for line in TRL_TRAIN_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
eval_rows = [json.loads(line) for line in TRL_EVAL_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]

for row in trl_rows:
    assert "prompt" in row and "chosen" in row and "rejected" in row
    assert "preference_reason" not in row
    assert row["chosen"][-1]["role"] == "assistant"
    assert row["rejected"][-1]["role"] == "assistant"

print(f"Validated {len(trl_rows)} TRL rows")
print(f"Train rows: {len(train_rows)}")
print(f"Eval rows:  {len(eval_rows)}")
print("Confirmed: preference_reason is not present in the DPO training rows.")
print(json.dumps(trl_rows[0], indent=2)[:1200])


## 4. Load datasets and configure DPO

This demo uses `HuggingFaceTB/SmolLM2-135M-Instruct` so the whole run can fit on a laptop. The same DPO structure scales to larger LLMs.

Key settings:

- `beta`: controls how strongly DPO pushes away from the rejected answer relative to the base model.
- LoRA settings: train a lightweight adapter instead of updating every model weight.
- train/eval split: train on 72 pairs, then score the 8 held-out pairs for a cleaner before/after.


In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("json", data_files=str(TRL_TRAIN_PATH), split="train")
eval_dataset = load_dataset("json", data_files=str(TRL_EVAL_PATH), split="train")
print(train_dataset)
print(eval_dataset)


In [ ]:
# This is the base LLM. DPO will update a LoRA adapter on top of this model.
# Default is Colab-friendly. For a stronger GPU, try: Qwen/Qwen2.5-1.5B-Instruct
BASE_MODEL = os.environ.get("HF_DPO_BASE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
OUTPUT_DIR = os.environ.get("HF_DPO_OUTPUT_DIR", "outputs/gen-academy-brand-voice-hf-dpo-qwen05b")

# These settings are designed for a realistic Colab run on the expanded 360-pair dataset.
# Full reversal is much more likely with a stronger model, length-normalized evaluation,
# and enough epochs for the adapter to separate chosen from rejected responses.
dpo_settings = {
    "base_model": BASE_MODEL,
    "output_dir": OUTPUT_DIR,
    "beta": 0.1,
    "learning_rate": 5e-5,
    "num_train_epochs": 5,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "max_length": 1024,
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
}

dpo_settings


## 5. Before DPO: score preference pairs

To make the before/after concrete, we score both completions instead of relying on free-form generations.

For each pair, the notebook computes:

`margin = logprob(chosen response) - logprob(rejected response)`

- Positive margin: the model already prefers the Gen Academy-style answer.
- Negative margin: the model prefers the off-brand answer.
- After DPO, we want the margin to move upward.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def preferred_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = preferred_device()
MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Using device: {DEVICE}")
if DEVICE != "cuda":
    print("For a faster Colab run, choose Runtime > Change runtime type > T4 GPU or better.")


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


def load_base_model():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=MODEL_DTYPE)
    model.eval()
    if DEVICE != "cpu":
        model.to(DEVICE)
    return model, tokenizer


def completion_logprob(model, tokenizer, prompt_messages, completion_text):
    # Convert chat messages into the exact text format expected by the base LLM.
    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full_text = prompt_text + completion_text + (tokenizer.eos_token or "")
    prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    full = tokenizer(full_text, return_tensors="pt", add_special_tokens=False)
    input_ids = full["input_ids"]
    if DEVICE != "cpu":
        input_ids = input_ids.to(DEVICE)
    with torch.no_grad():
        logits = model(input_ids).logits
    labels = input_ids[:, 1:]
    logprobs = torch.log_softmax(logits[:, :-1, :], dim=-1)
    token_logprobs = logprobs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

    # Only score the assistant completion tokens, not the prompt tokens.
    start = max(prompt_ids.shape[-1] - 1, 0)
    completion_token_logprobs = token_logprobs[:, start:]
    token_count = int(completion_token_logprobs.numel())
    total_logprob = float(completion_token_logprobs.sum().detach().cpu())
    avg_logprob = total_logprob / max(token_count, 1)
    return total_logprob, avg_logprob, token_count


def score_pairs(model, tokenizer, rows, label):
    scored = []
    for row in rows:
        chosen = row["chosen"][-1]["content"]
        rejected = row["rejected"][-1]["content"]
        chosen_total, chosen_avg, chosen_tokens = completion_logprob(model, tokenizer, row["prompt"], chosen)
        rejected_total, rejected_avg, rejected_tokens = completion_logprob(model, tokenizer, row["prompt"], rejected)
        # Main metric: length-normalized margin. This avoids penalizing the better answer just because it is longer.
        normalized_margin = chosen_avg - rejected_avg
        raw_sum_margin = chosen_total - rejected_total
        scored.append(
            {
                "model": label,
                "id": row["id"],
                "split": row["split"],
                "category": row["category"],
                "chosen_logp_sum": chosen_total,
                "rejected_logp_sum": rejected_total,
                "chosen_logp_avg": chosen_avg,
                "rejected_logp_avg": rejected_avg,
                "chosen_tokens": chosen_tokens,
                "rejected_tokens": rejected_tokens,
                "raw_sum_margin": raw_sum_margin,
                "margin": normalized_margin,
                "prefers_chosen": normalized_margin > 0,
            }
        )
    return pd.DataFrame(scored)


base_model, tokenizer = load_base_model()
base_all_scores = score_pairs(base_model, tokenizer, trl_rows, "base")
base_eval_scores = base_all_scores[base_all_scores["split"] == "eval"].copy()
display(base_eval_scores[["id", "category", "margin", "raw_sum_margin", "prefers_chosen"]])
print("Base eval normalized win rate:", f"{base_eval_scores['prefers_chosen'].mean():.0%}")
del base_model
clear_memory()


## 6. Train the DPO adapter

This is the actual LLM DPO step. `DPOTrainer` compares each chosen/rejected pair under the current LLM and updates the LoRA adapter so chosen responses become more likely relative to rejected responses.

The cell trains only if `OUTPUT_DIR` does not exist. That keeps repeated notebook runs from retraining accidentally. To retrain from scratch, delete the output folder or change `HF_DPO_OUTPUT_DIR`.


In [ ]:
# If the adapter already exists, skip retraining and reuse it for the before/after comparison.
RUN_HF_DPO_TRAINING = not Path(OUTPUT_DIR).exists()

if RUN_HF_DPO_TRAINING:
    import shutil
    from peft import LoraConfig
    from trl import DPOConfig, DPOTrainer

    if Path(OUTPUT_DIR).exists():
        shutil.rmtree(OUTPUT_DIR)

    # DPOConfig tells TRL how strongly to optimize preference separation.
    training_args = DPOConfig(
        output_dir=OUTPUT_DIR,
        beta=dpo_settings["beta"],
        learning_rate=dpo_settings["learning_rate"],
        num_train_epochs=dpo_settings["num_train_epochs"],
        per_device_train_batch_size=dpo_settings["per_device_train_batch_size"],
        gradient_accumulation_steps=dpo_settings["gradient_accumulation_steps"],
        max_length=dpo_settings["max_length"],
        logging_steps=5,
        save_strategy="epoch",
        report_to=[],
        fp16=(DEVICE == "cuda"),
        model_init_kwargs={"torch_dtype": MODEL_DTYPE},
    )

    # LoRA keeps the project lightweight by training adapter weights instead of the full LLM.
    peft_config = LoraConfig(
        r=dpo_settings["lora_r"],
        lora_alpha=dpo_settings["lora_alpha"],
        lora_dropout=dpo_settings["lora_dropout"],
        target_modules="all-linear",
        task_type="CAUSAL_LM",
    )

    # DPOTrainer consumes prompt/chosen/rejected rows. It does not consume preference_reason.
    trainer = DPOTrainer(
        model=BASE_MODEL,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=peft_config,
    )
    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    del trainer
    clear_memory()
else:
    print(f"Adapter already exists at {OUTPUT_DIR}. Delete that folder or change HF_DPO_OUTPUT_DIR to retrain from scratch.")


## 7. Training curves

These plots turn the DPO math into something easier to narrate.

- DPO loss should generally move down.
- Reward margin should generally move up, meaning the trainer is separating chosen answers from rejected answers.
- Reward accuracy shows how often the chosen response receives the higher implicit reward in a batch.


In [ ]:
trainer_states = sorted(
    Path(OUTPUT_DIR).glob("checkpoint-*/trainer_state.json"),
    key=lambda path: int(path.parent.name.split("-")[-1]),
)
trainer_state_path = trainer_states[-1] if trainer_states else None
if trainer_state_path:
    trainer_state = json.loads(trainer_state_path.read_text())
    logs = pd.DataFrame(trainer_state["log_history"])
    metric_logs = logs[logs["loss"].notna()].copy()
    display(metric_logs[["step", "loss", "rewards/margins", "rewards/accuracies"]])

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(metric_logs["step"], metric_logs["loss"], marker="o", color="#315c72")
    axes[0].set_title("DPO Loss")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("loss")
    axes[1].plot(metric_logs["step"], metric_logs["rewards/margins"], marker="o", color="#7c4d1d")
    axes[1].axhline(0, color="#777", linewidth=1)
    axes[1].set_title("Reward Margin")
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("chosen - rejected")
    plt.tight_layout()
else:
    print("No trainer_state.json found yet. Run training first.")


## 8. After DPO: score the same pairs again

This cell reloads the base LLM, attaches the trained LoRA adapter, and repeats the exact same scoring function.

The important comparison is not just whether the tuned model wins every pair. For a small model and tiny dataset, the clearer signal is whether the preference margin moves in the right direction on train and held-out examples.


In [ ]:
from peft import PeftModel

# Load the same base LLM, then attach the DPO-trained LoRA adapter.
tuned_base, tokenizer = load_base_model()
tuned_model = PeftModel.from_pretrained(tuned_base, OUTPUT_DIR)
tuned_model.eval()
if DEVICE != "cpu":
    tuned_model.to(DEVICE)

# Score the exact same chosen/rejected completions again after DPO.
tuned_all_scores = score_pairs(tuned_model, tokenizer, trl_rows, "tuned")
comparison_all = base_all_scores.merge(
    tuned_all_scores,
    on=["id", "split", "category"],
    suffixes=("_base", "_tuned"),
)
comparison_all["margin_change"] = comparison_all["margin_tuned"] - comparison_all["margin_base"]
comparison_eval = comparison_all[comparison_all["split"] == "eval"].copy()

def summarize(scope, frame):
    return {
        "scope": scope,
        "pairs": len(frame),
        "base_win_rate": frame["prefers_chosen_base"].mean(),
        "tuned_win_rate": frame["prefers_chosen_tuned"].mean(),
        "avg_base_margin": frame["margin_base"].mean(),
        "avg_tuned_margin": frame["margin_tuned"].mean(),
        "avg_margin_change": frame["margin_change"].mean(),
        "pairs_moved_toward_chosen": int((frame["margin_change"] > 0).sum()),
        "pct_moved_toward_chosen": (frame["margin_change"] > 0).mean(),
        "pairs_flipped_to_chosen": int(((~frame["prefers_chosen_base"]) & frame["prefers_chosen_tuned"]).sum()),
    }

# margin_change > 0 means DPO moved the LLM toward the Gen Academy-style answer.
summary = pd.DataFrame([summarize("eval", comparison_eval), summarize("all_pairs", comparison_all)])
print(
    f"DPO moved the normalized preference margin toward the chosen response on "
    f"{int((comparison_all['margin_change'] > 0).sum())}/{len(comparison_all)} total pairs "
    f"and {int((comparison_eval['margin_change'] > 0).sum())}/{len(comparison_eval)} eval pairs."
)
print(
    f"Tuned normalized win rate: {comparison_eval['prefers_chosen_tuned'].mean():.0%} on eval, "
    f"{comparison_all['prefers_chosen_tuned'].mean():.0%} overall."
)
display(summary)
display(comparison_eval[["id", "category", "margin_base", "margin_tuned", "margin_change", "prefers_chosen_base", "prefers_chosen_tuned"]])

plot_df = comparison_eval.sort_values("margin_change")
fig, ax = plt.subplots(figsize=(9, 8))
y = range(len(plot_df))
ax.barh([i - 0.18 for i in y], plot_df["margin_base"], height=0.35, label="base", color="#a9b3bb")
ax.barh([i + 0.18 for i in y], plot_df["margin_tuned"], height=0.35, label="tuned", color="#315c72")
ax.axvline(0, color="#666", linewidth=1)
ax.set_yticks(list(y))
ax.set_yticklabels(plot_df["id"] + " / " + plot_df["category"], fontsize=8)
ax.set_title("Held-Out Length-Normalized Preference Margins")
ax.set_xlabel("chosen avg logprob - rejected avg logprob")
ax.legend()
plt.tight_layout()


## 9. Concrete before/after example

The notebook automatically selects a pair where the base model preferred the rejected response and the tuned adapter prefers the chosen response. If no pair fully flips, it selects the pair with the largest positive margin shift.

In [ ]:
# Prefer a true flip because it is the clearest before/after example.
flipped = comparison_all[
    (~comparison_all["prefers_chosen_base"]) & (comparison_all["prefers_chosen_tuned"])
].sort_values("margin_change", ascending=False)
if len(flipped):
    example_id = flipped.iloc[0]["id"]
else:
    example_id = comparison_all.sort_values("margin_change", ascending=False).iloc[0]["id"]

# Pull the original human-readable row so we can show the actual text, not just metrics.
example_source = source_df[source_df["id"] == example_id].iloc[0]
example_scores = comparison_all[comparison_all["id"] == example_id].iloc[0]

print("Concrete before/after example")
print(f"Example: {example_id} / {example_source['category']}")
print(f"Base normalized margin:  {example_scores['margin_base']:.3f}")
print(f"Tuned normalized margin: {example_scores['margin_tuned']:.3f}")
print(f"Change:                  {example_scores['margin_change']:.3f}")
print()
print("TASK")
print(example_source["prompt"])
print()
print("ON-BRAND CHOSEN")
print(example_source["chosen"])
print()
print("OFF-BRAND REJECTED")
print(example_source["rejected"])

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["base", "tuned"], [example_scores["margin_base"], example_scores["margin_tuned"]], color=["#a9b3bb", "#315c72"])
ax.axhline(0, color="#666", linewidth=1)
ax.set_title(f"Normalized Margin Shift: {example_id}")
ax.set_ylabel("chosen avg logprob - rejected avg logprob")
plt.tight_layout()
